In [3]:
import pandas as pd
import numpy as np
import gc
from IPython.display import display
from mnp.ingestion.loader import load_endes
from mnp.utils.profiler import centrar_notebook
from mnp.utils.cleaning import run_phase1_engine
# Configuración visual
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
centrar_notebook()

print("[OK]")

2026-06-13 23:14:47.222 | INFO     | mnp.config:<module>:11 - PROJ_ROOT path is: /Users/abelguevarah/Desktop/invs/malnutrition-research


[OK]


In [4]:
# Carga de datos
history = load_endes(
        year=range(2007, 2025),
        module="housing",
        record="rech23h",
        meta=True
    )

dfs = []
col_labels_hist = {}
val_labels_hist = {}

for year in sorted(list(history.keys())):
    year_df, year_meta = history.pop(year)
    col_labels_hist[year] = year_meta.column_names_to_labels
    val_labels_hist[year] = year_meta.variable_value_labels
    
    dfs.append(year_df.assign(year=year))

df_raw = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f"[OK] Data RECH23 cargada: {len(df_raw)} filas y {df_raw.shape[1]} columnas.")
print(f"[OK] Diccionario cargado: {len(col_labels_hist)} Column Labels, {len(val_labels_hist)} Value Labels")

2026-06-13 23:14:49.005 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH23.SAV desde Modulo65 [Alias: rech23h]
2026-06-13 23:14:49.491 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH23.SAV desde Modulo65 [Alias: rech23h]
2026-06-13 23:14:50.215 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH23.SAV desde Modulo65 [Alias: rech23h]
2026-06-13 23:14:50.933 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH23.SAV desde Modulo65 [Alias: rech23h]
2026-06-13 23:14:51.452 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH23.sav desde Modulo65 [Alias: rech23h]
2026-06-13 23:14:51.963 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH23.sav desde Modulo65 [Alias: rech23h]
2026-06-13 23:14:52.489 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH23.SAV desde Modulo65 [Alias: rech23h]
2026-06-13 23:14:53.036 | I

In [5]:
# Inspeccionar variable
print(df_raw['SH107'].value_counts(dropna=False))
print(df_raw['SH124'].value_counts(dropna=False))
print(df_raw['SH224'].value_counts(dropna=False))
print(df_raw['HV234'].value_counts(dropna=False))

SH107
NaN    580453
4.0     15261
3.0      6745
2.0      1829
5.0      1681
9.0       702
1.0       616
Name: count, dtype: int64
SH124
NaN    580682
4.0     16432
3.0      4802
5.0      1998
2.0      1891
1.0       802
9.0       680
Name: count, dtype: int64
SH224
4.0    225521
NaN    163628
9.0    111852
3.0     53276
2.0     22299
1.0     16697
5.0     14014
Name: count, dtype: int64
HV234
30.0     241953
994.0    114213
NaN      110891
15.0      64823
995.0     31273
7.0       26019
0.0       18115
Name: count, dtype: int64


In [6]:
# Inspeccionar variable
print(df_raw['SHSEMES'].value_counts(dropna=False))
print(df_raw['SHTRIMES'].value_counts(dropna=False))

SHSEMES
2.0    272128
1.0    270789
NaN     64370
Name: count, dtype: int64
SHTRIMES
NaN     542917
22.0      8065
11.0      5138
4.0       4966
9.0       4958
8.0       4923
10.0      4913
7.0       4900
5.0       4637
12.0      4631
6.0       4458
21.0      3737
3.0       2293
2.0       2200
1.0       1884
14.0      1586
13.0      1081
Name: count, dtype: int64


In [ ]:
df_raw.head(10)

In [ ]:
import numpy as np
from mnp.utils.cleaning import run_phase1_engine
from mnp.configs.rech23 import config_f1

# --- 1. CURA MASIVA PARA EL DICCIONARIO CORRUPTO DE YODO ---
# Convertimos las categorías 1-4 a PPM puros.
# El 5 y 9 se van a la zona de purga (995 y 994) para que Fase 1 los vuelva NaN.
diccionario_yodo = {
    1.0: 0.0, 2.0: 7.0, 3.0: 15.0, 4.0: 30.0,
    5.0: 995.0, 9.0: 994.0
}
columnas_yodo = ['SH107', 'SH124', 'SH224']
for col in columnas_yodo:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].replace(diccionario_yodo)


# --- 2. SPLITS MANUALES POR COLUMNAS RECICLADAS DEL INEI ---
# Agua y Basura
df_raw['SH42_Agua_Todo_El_Dia'] = np.where(df_raw['year'] >= 2010, df_raw['SH42'], np.nan)
df_raw['SH48_Conserva_Agua'] = np.where(df_raw['year'] >= 2010, df_raw['SH48'], np.nan)
df_raw['SH48_Basura'] = np.where(df_raw['year'] == 2009, df_raw['SH48'], np.nan)

# Limpieza, Basura y Luz (SH60)
df_raw['SH56_Frec_Limpieza'] = np.where(df_raw['year'] >= 2010, df_raw['SH56'], np.nan)
df_raw['SH56_Limpió_Baño'] = np.where(df_raw['year'] == 2009, df_raw['SH56'], np.nan)
df_raw['SH59_Frec_Recojo'] = np.where(df_raw['year'] >= 2010, df_raw['SH59'], np.nan)
df_raw['SH59_Recogen_Basura'] = np.where(df_raw['year'] == 2009, df_raw['SH59'], np.nan)
df_raw['SH60_Tipo_Basurero'] = np.where(df_raw['year'] >= 2010, df_raw['SH60'], np.nan)
df_raw['SH60_Fuente_Luz'] = np.where(df_raw['year'] == 2009, df_raw['SH60'], np.nan)

# Split de Luz y Hectáreas (SH70)
df_raw['SH70_Fuente_Luz'] = np.where(df_raw['year'] >= 2010, df_raw['SH70'], np.nan)
df_raw['SH70_Hectareas'] = np.where(df_raw['year'] == 2009, df_raw['SH70'], np.nan)


# Drop
df_raw = df_raw.drop(columns=['SH42', 'SH48', 'SH56', 'SH59', 'SH60', 'SH70'])

df_clean = run_phase1_engine(df_raw, config_f1)

print(f'[OK] Limpieza biológica completa. Shape: {df_clean.shape}')
df_clean.head(5)

In [ ]:
from importlib import reload
import mnp.utils.cleaning
reload(mnp.utils.cleaning)
from mnp.utils.cleaning import apply_standard_labels
from mnp.configs.rech23 import config_f1, config_f3

# 1. El motor estandariza todas las variables normales leyendo config_f3
df_final = apply_standard_labels(df_clean, val_labels_hist, config_f3, config_f1)

# 2. Mapeos manuales EXCLUSIVOS para las columnas del Split (que el diccionario no reconoce)
dict_binario = {1.0: "Sí", 2.0: "No", 0.0: "No"}
columnas_manuales = [
    'SH42_Agua_Todo_El_Dia', 
    'SH48_Conserva_Agua', 
    'SH56_Limpió_Baño', 
    'SH59_Recogen_Basura'
]

for col in columnas_manuales:
    if col in df_final.columns:
        df_final[col] = df_final[col].map(dict_binario)

print(f"[OK] Fase 3 (Estandarización de Etiquetas) completada.")
df_final.head(15)


In [ ]:
from mnp.config import INTERIM_DATA_DIR
import os
# Aseguramos que la carpeta interim exista
os.makedirs(INTERIM_DATA_DIR, exist_ok=True)
# Ruta del archivo final
output_path = INTERIM_DATA_DIR / "rech23_cleaned.parquet"
# Guardamos en formato parquet (preserva tipos de datos y ahorra RAM)
df_final.to_parquet(output_path, index=False)
print(f"[OK] Datos RECH23 (Fase 1, 2 y 3) guardados exitosamente en:")
print(f"-> {output_path}")

In [ ]:
import sweetviz as sv
from mnp.configs.column_labels import SWEETVIZ_LABELS
from mnp.config import INTERIM_DATA_DIR  # Importamos tu variable mágica

df_sweetviz = df_final.rename(columns=SWEETVIZ_LABELS)
reporte_sv = sv.analyze(df_sweetviz)

# Usamos la ruta absoluta inteligente y la pasamos a string
ruta_reporte = str(INTERIM_DATA_DIR / 'rech23_sweetviz_report.html')

reporte_sv.show_html(ruta_reporte)
reporte_sv.show_notebook(w="100%", h="800", filepath=ruta_reporte)


In [ ]:
df_final['HV270'].value_counts(dropna=False)